# Day 3
# Baseline models and the structural dataset

**Goal:** train and evaluate classical baselines on the tabular features, then describe the
same complexes structurally: fingerprints, PCA + clustering, and ligand analysis.

**Source paper:** [Redox potential prediction of Fe(ii)/Fe(iii) complexes: a density functional theory and graph neural network approach](https://pubs.rsc.org/dd/article/5/3/1098/311651/Redox-potential-prediction-of-Fe-ii-Fe-iii )

## Learning objectives

- Train Random Forest and Gaussian Process Regression models on engineered features.
- Evaluate models using RMSE, MAE, and R2, and read a parity plot.
- Inspect feature importances from Random Forest and uncertainties from GPR.
- Encode a complex as a Morgan fingerprint and work out what a set bit means.
- Project the fingerprint space with PCA and cluster it with K-Means.
- Split complexes into ligands and sort those ligands into chemical classes.
- Describe a molecule as a graph: nodes, edges, node features, adjacency matrix.

---


### Set `REPO_ROOT`

Every notebook in this workshop locates the repository the same way: by walking up
from the notebook's own directory until it finds one containing both `data/` and
`notebooks/`.

Run this cell before any other code cell, and check that the printed path is your clone.


In [ ]:
# --- Set REPO_ROOT --------------------------------------------------------
# Locate the repository root by searching upward from this notebook's directory.
from pathlib import Path


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for d in (start, *start.parents):
        if (d / "data").is_dir() and (d / "notebooks").is_dir():
            return d
    raise FileNotFoundError(f"Repo root not found above {start}")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
FIG_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print("Repo root: ", REPO_ROOT)
print("Data dir:  ", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Figure dir:", FIG_DIR)



## Setup

### Imports


In [ ]:
import os
import time
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
import seaborn as sns
import networkx as nx

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw
from IPython.display import display

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='notebook')

# RDKit logs a line every time a bond to the metal is broken; that is tens of
# thousands of lines over the full dataset, so silence it
RDLogger.DisableLog('rdApp.info')



### Loading the dataset

Day 1 finished by writing the cleaned dataset to `output/data_cleaned.pkl`. That is the
same 1546 complexes used here, so rather than rebuild it, load it back once and use it for
the whole notebook. 

`csd_code` is stored on the index, so `reset_index()` puts it back as a column.


In [ ]:
# CLEANED_PATH = Path(os.environ.get('SCRATCH', '.')) / 'Fe-Redox-GNN' / 'data_cleaned.csv'
CLEANED_PATH = OUTPUT_DIR / 'data_cleaned.pkl'
df = pd.read_pickle(CLEANED_PATH)
df = df.reset_index()   # csd_code is stored as the index in this file

print('Loaded cleaned data with', len(df), 'rows from', CLEANED_PATH)
display(df.head())


---

## Part 1 - Baseline machine learning

Sections 1.1-1.3 are the theoretical background; the models are trained from 1.4 onwards.

### 1.1 Random Forest Regression

A Random Forest is an ensemble of decision trees trained on bootstrap samples and random feature subsets. At a split, the objective for a candidate split can be written in terms of the mean-squared error (MSE):

$$
\mathrm{MSE}_{\text{split}} = \frac{n_L}{n} \mathrm{MSE}_L + \frac{n_R}{n} \mathrm{MSE}_R
$$

The ensemble prediction is the average of individual tree predictions:

$$
\widehat{y} = \frac{1}{B} \sum_{b=1}^B T_b(\mathbf{x})
$$

Advantages: handles non-linear relationships, robust to outliers, provides feature importance. Disadvantages: cannot extrapolate beyond training data and does not provide uncertainty estimates.

### 1.2 Gaussian Process Regression (GPR)

A Gaussian Process is a distribution over functions specified by a mean function $m(\mathbf{x})$ and a kernel $k(\mathbf{x}, \mathbf{x}')$:

$$
f(\mathbf{x}) \sim \mathcal{GP}(m(\mathbf{x}), k(\mathbf{x}, \mathbf{x}'))
$$

The Radial Basis Function (RBF) kernel is commonly used:

$$
k(\mathbf{x}, \mathbf{x}') = \sigma_f^2 \exp\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\right)
$$

Given training data $(X, y)$ and a test point $\mathbf{x}_\ast$, the predictive mean and variance are:

$$
\widehat{y}_\ast = \mathbf{k}_\ast^\top (K + \sigma_n^2 I)^{-1} \mathbf{y}
$$

$$
\mathrm{Var}(\widehat{y}_\ast) = k(\mathbf{x}_\ast, \mathbf{x}_\ast) - \mathbf{k}_\ast^\top (K + \sigma_n^2 I)^{-1} \mathbf{k}_\ast
$$

GPR provides uncertainty estimates but scales as $\mathcal{O}(n^3)$, so it is most practical for small datasets.

### 1.3 Evaluation metrics

Root Mean Squared Error (RMSE):

$$
\mathrm{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^n (y_i - \widehat{y}_i)^2}
$$

Coefficient of determination ($R^2$):

$$
R^2 = 1 - \frac{\sum_{i=1}^n (y_i - \widehat{y}_i)^2}{\sum_{i=1}^n (y_i - \overline{y})^2}
$$

Mean Absolute Error (MAE):

$$
\mathrm{MAE} = \frac{1}{n} \sum_{i=1}^n |y_i - \widehat{y}_i|
$$



### 1.4 Features, split, and scaling


In [ ]:
TARGET_COL = 'redox_pot'
if TARGET_COL not in df.columns:
    raise RuntimeError(f"Target column '{TARGET_COL}' not found in the cleaned dataset")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
FEATURE_COLS = [
    c for c in numeric_cols
    if c not in {TARGET_COL, 'q_sum', 'is_outlier_any'} and df[c].nunique() > 1
]

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

print(f"Dataset: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Target: {TARGET_COL}  Range: [{y.min():.4f}, {y.max():.4f}]  Mean: {y.mean():.4f} ± {y.std():.4f}")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

# Standardize features (fit on training data only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### 1.5 Helper functions: evaluation and parity plot


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def evaluate_model(y_true, y_pred, model_name='Model'):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{model_name} Performance:\n  RMSE: {rmse:.4f}\n  MAE:  {mae:.4f}\n  R²:   {r2:.4f}")
    return {'RMSE': rmse, 'MAE': mae, 'R²': r2}

import matplotlib.pyplot as plt

def parity_plot(y_true, y_pred, model_name='Model', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true, y_pred, alpha=0.5, s=30, edgecolors='gray', linewidth=0.3, color='steelblue')
    lims = [min(np.min(y_true), np.min(y_pred)), max(np.max(y_true), np.max(y_pred))]
    margin = (lims[1] - lims[0]) * 0.05
    lims = [lims[0] - margin, lims[1] + margin]
    ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel(f'Actual {TARGET_COL}')
    ax.set_ylabel(f'Predicted {TARGET_COL}')
    ax.set_title(model_name)
    ax.legend()
    ax.set_aspect('equal')
    return ax


### 1.6 Training a baseline Random Forest


In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, max_depth=None, min_samples_split=2,
                                 min_samples_leaf=1, max_features='sqrt', random_state=42, n_jobs=-1)
import time
start_time = time.time()
rf_model.fit(X_train_scaled, y_train)
train_time = time.time() - start_time
print(f'Training time: {train_time:.2f} seconds')

# Predictions
y_train_pred_rf = rf_model.predict(X_train_scaled)
y_test_pred_rf = rf_model.predict(X_test_scaled)

train_metrics_rf = evaluate_model(y_train, y_train_pred_rf, 'Random Forest (Train)')
test_metrics_rf = evaluate_model(y_test, y_test_pred_rf, 'Random Forest (Test)')

# Overfitting check
gap = train_metrics_rf['R²'] - test_metrics_rf['R²']
print(f'Overfitting gap (Train R² - Test R²): {gap:.4f}')


### 1.7 Parity plots for Random Forest


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
parity_plot(y_train, y_train_pred_rf, 'Random Forest (Train)', ax=axes[0])
parity_plot(y_test, y_test_pred_rf, 'Random Forest (Test)', ax=axes[1])
plt.suptitle('Random Forest Regression: Parity Plots', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'rf_parity_plots.png', dpi=150, bbox_inches='tight')
plt.show()


### 1.8 Feature importance

A trained forest can be asked which of the input columns it actually used. That is what
`rf_model.feature_importances_` returns: one number per entry of `FEATURE_COLS`, all
non-negative and summing to 1, plotted here as a horizontal bar chart.

**How the number is computed.** Every split in every tree is chosen to reduce the impurity
of the node it splits. For regression the impurity is the mean squared error of the target
values sitting in that node, and the reduction achieved by a split of node $t$ into
children $L$ and $R$ is

$$
\Delta_t = \mathrm{MSE}_t - \frac{n_L}{n_t}\mathrm{MSE}_L - \frac{n_R}{n_t}\mathrm{MSE}_R
$$

That reduction is credited to whichever feature was used at that split, weighted by how
much of the data reached the node, $n_t / n$. Summing over every node in a tree that split
on feature $f$, averaging over the $B = 100$ trees, and normalising so the values add to 1
gives

$$
\mathrm{Imp}(f) = \frac{1}{B}\sum_{b=1}^{B} \sum_{t \in b\,:\,\text{split on } f} \frac{n_t}{n}\, \Delta_t
$$

This is the quantity the x-axis calls **mean decrease in impurity** (MDI). A feature scores
highly when it is chosen often, chosen near the root where $n_t$ is large, and produces
large drops in MSE when it is chosen.

**How to read it, and how not to.**

- The values are relative and unitless. A bar twice as long means the feature accounted for
  twice as much impurity reduction, not that the redox potential is twice as sensitive to it.
- It is computed on the **training** data, from the structure of the fitted trees. It
  describes what this forest did, not what is physically true. A feature can be important to
  the model and still be a proxy for something else.
- **Correlated features split the credit.** If two columns carry the same information, the
  trees use them interchangeably and each ends up with roughly half the importance, so a
  genuinely relevant feature can look weak. Read the Part 4 correlation work alongside this
  plot before concluding a feature does not matter.
- MDI is biased toward features with many distinct values, because a continuous column
  offers more candidate split points than a near-binary one such as `n_S`.
- `max_features='sqrt'` means each split only sees a random subset of the columns, so a
  feature can be omitted from a split simply because it was not offered. Averaging over 100
  trees smooths this out but does not remove it.
- Scaling does not affect this. Trees split on thresholds, so `X_train_scaled` produces the
  same tree structure, and the same importances, as the unscaled features would.

A stricter alternative is permutation importance: shuffle one column of the **test** set,
re-score the model, and record how much performance drops. That measures the effect on
held-out predictions rather than on training-set impurity, at the cost of one refit-free
evaluation per column.


In [ ]:
importances = rf_model.feature_importances_
importance_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances}).sort_values('Importance', ascending=True)
fig, ax = plt.subplots(figsize=(8, max(4, len(FEATURE_COLS) * 0.4)))
ax.barh(importance_df['Feature'], importance_df['Importance'], color='teal', edgecolor='white', alpha=0.85)
ax.set_xlabel('Feature Importance (Mean Decrease in Impurity)')
ax.set_title('Random Forest Feature Importance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
for _, row in importance_df.tail(5).iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")


### 1.9 Training a Gaussian Process Regression model


In [ ]:
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-3, 1e3)) + WhiteKernel(1e-2)
max_gpr_samples = 1000
if X_train_scaled.shape[0] > max_gpr_samples:
    print(f'Training set ({X_train_scaled.shape[0]}) is large for GPR, subsampling to {max_gpr_samples}')
    rng = np.random.RandomState(42)
    idx = rng.choice(X_train_scaled.shape[0], max_gpr_samples, replace=False)
    X_train_gpr = X_train_scaled[idx]
    y_train_gpr = y_train[idx]
else:
    X_train_gpr = X_train_scaled
    y_train_gpr = y_train

from sklearn.gaussian_process import GaussianProcessRegressor

gpr_model = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, alpha=1e-6, random_state=42)
start_time = time.time()
gpr_model.fit(X_train_gpr, y_train_gpr)
train_time = time.time() - start_time
print(f'GPR training time: {train_time:.2f} seconds')
print('Optimized kernel:', gpr_model.kernel_)
print('Log-marginal-likelihood:', gpr_model.log_marginal_likelihood_value_)


### 1.10 Predictions with uncertainty

`gpr_model.predict(X_test_scaled, return_std=True)` returns two arrays instead of one: the
predicted value and a standard deviation $\sigma$ for each test complex. The Random Forest
in 1.6 could only return the first of these.

**What $\sigma$ is.** A Gaussian process does not predict a number, it predicts a
distribution. For a test point $\mathbf{x}_\ast$ the posterior is Gaussian with the mean and
variance written out in 1.2, and $\sigma = \sqrt{\mathrm{Var}(\widehat{y}_\ast)}$ is the
width of that Gaussian. It answers "how far from my own prediction would I not be surprised
to find the truth", in volts, the same units as the target. The error bars in the middle
panel are $\pm 2\sigma$, so roughly 95% of the test points should fall inside their own bar
if the model is honest.

**Where it comes from.** Two things make $\sigma$ large:

- **Distance from the training data.** The RBF kernel measures similarity, so a complex that
  resembles nothing in the training set has $\mathbf{k}_\ast \approx \mathbf{0}$, the
  subtracted term in the variance formula vanishes, and $\sigma$ rises to the prior width.
  This is the part that shrinks if you collect more data in that region.
- **Noise in the labels.** `WhiteKernel` is part of the fitted kernel, so the optimiser
  estimates how much scatter the targets carry that no smooth function of these features can
  explain. That floor is present in every prediction and does not shrink with more data.

Note that $\sigma$ depends only on **where** the test point sits in feature space. It is
computed from `X_test_scaled` alone; the true `y_test` values play no part in it. The model
can therefore flag a prediction as unreliable before any reference value exists, which is
what makes this useful: run DFT on the complexes where $\sigma$ is largest, and trust the
cheap prediction where it is small.

**Caveats.**

- $\sigma$ is not the error. It is a claim about the error, conditional on the kernel being
  an appropriate description of the data. Check it: count how many test points fall outside
  their $\pm 2\sigma$ bar. Far more than 5% means the model is overconfident, far fewer
  means it is hedging.
- The uncertainties are relative to the 1000 training complexes the GPR was fitted on in
  1.9, not to the full training set.
- The third panel is the distribution of $\sigma$ across the test set. A tight spike means
  the model finds the whole test set equally familiar, which is expected here because the
  split was random; a long right tail would mark a subset of complexes worth inspecting.


In [ ]:
y_test_pred_gpr, y_test_std_gpr = gpr_model.predict(X_test_scaled, return_std=True)
y_train_pred_gpr, y_train_std_gpr = gpr_model.predict(X_train_gpr, return_std=True)

train_metrics_gpr = evaluate_model(y_train_gpr, y_train_pred_gpr, 'GPR (Train)')
test_metrics_gpr = evaluate_model(y_test, y_test_pred_gpr, 'GPR (Test)')

# Parity + uncertainty plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
# RF parity
parity_plot(y_test, y_test_pred_rf, 'Random Forest', ax=axes[0])
# GPR parity with error bars
axes[1].errorbar(y_test, y_test_pred_gpr, yerr=2 * y_test_std_gpr, fmt='o', alpha=0.4, markersize=4,
                 color='steelblue', ecolor='lightblue', elinewidth=1, capsize=2)
lims = [min(y_test.min(), y_test_pred_gpr.min()), max(y_test.max(), y_test_pred_gpr.max())]
margin = (lims[1] - lims[0]) * 0.05
lims = [lims[0] - margin, lims[1] + margin]
axes[1].plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_xlabel(f'Actual {TARGET_COL}')
axes[1].set_ylabel(f'Predicted {TARGET_COL}')
axes[1].set_title('GPR Predictions with 95% CI', fontweight='bold')
axes[1].legend()
axes[1].set_aspect('equal')

# Uncertainty distribution
axes[2].hist(y_test_std_gpr, bins=30, color='coral', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Predicted Standard Deviation')
axes[2].set_ylabel('Count')
axes[2].set_title('Distribution of Prediction Uncertainty', fontweight='bold')
axes[2].axvline(np.mean(y_test_std_gpr), color='red', linestyle='--', label=f'Mean σ = {np.mean(y_test_std_gpr):.4f}')
axes[2].legend()

plt.suptitle('GPR Results and Comparison on the Test set', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'gpr_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('Uncertainty statistics: mean σ =', np.mean(y_test_std_gpr))


### 1.11 Model comparison


In [ ]:
results = pd.DataFrame({
    'Model': ['Random Forest', 'GPR'],
    'RMSE (Test)': [test_metrics_rf['RMSE'], test_metrics_gpr['RMSE']],
    'MAE (Test)': [test_metrics_rf['MAE'], test_metrics_gpr['MAE']],
    'R² (Test)': [test_metrics_rf['R²'], test_metrics_gpr['R²']],
    'R² (Train)': [train_metrics_rf['R²'], train_metrics_gpr['R²']],
})

print('Model Comparison Summary:')
display(results)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].bar(results['Model'], results['RMSE (Test)'], color=['steelblue', 'coral'], edgecolor='black', alpha=0.85)
axes[0].set_ylabel('RMSE')
axes[0].set_title('Test RMSE (lower is better)', fontweight='bold')
for i, v in enumerate(results['RMSE (Test)']):
    axes[0].text(i, v + 0.001, f'{v:.4f}', ha='center', fontweight='bold')

axes[1].bar(results['Model'], results['R² (Test)'], color=['steelblue', 'coral'], edgecolor='black', alpha=0.85)
axes[1].set_ylabel('R²')
axes[1].set_title('Test R² (higher is better)', fontweight='bold')
axes[1].set_ylim(0, 1.05)
for i, v in enumerate(results['R² (Test)']):
    axes[1].text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')

plt.suptitle('Classical ML Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results

BASELINE_PATH = OUTPUT_DIR / 'baseline_results.csv'
#df = pd.read_csv(CLEANED_PATH)

#results.to_csv(os.path.join(WORKSHOP_PATH, 'baseline_results.csv'), index=False)
results.to_csv(BASELINE_PATH, index=False)
print('Saved baseline_results.csv for Day 4 comparison')



**Mentor checkpoint 6**: after the baseline models (Part 1)

- Which model performed better on the test set and why?
- Is there evidence of overfitting in either model?
- What advantage do GPR uncertainty estimates provide in experimental design?
- What performance improvement would justify using a GNN over these baselines?

Proceed only after confirmation.

---


---

## Part 2 - The redox potential distribution

The baselines above were trained on the tabular features. The rest of the notebook describes
the same complexes structurally instead, starting from the quantity every model here is
trying to predict: `redox_pot` is the redox potential in volts, one value per complex.
Everything that follows is an attempt to explain the shape of this curve.


In [ ]:
plt.figure(figsize=(9.5, 6), dpi=150)

ax = sns.histplot(df['redox_pot'], kde=True, bins=30)

plt.xlabel('Redox potential (V)', fontsize=35)
plt.xlim([-2, 3])
plt.xticks(fontsize=25)

plt.ylabel('Count', fontsize=35)
plt.ylim([0, 200])
plt.yticks(fontsize=25)

ax.yaxis.set_major_locator(plt.MaxNLocator(4))
ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.tick_params(which='major', length=8, width=2)
ax.tick_params(axis='x', which='minor', length=5, width=1.5)

plt.tight_layout()
plt.show()


## Part 3 - Morgan fingerprints

We now have a problem: molecules are naturally represented as **graphs**, but most machine-learning models, including the scikit-learn models we will use later, expect a fixed-size table of numbers.

For example, one molecule might contain 10 atoms while another contains 30 atoms. We cannot simply put the atom features side-by-side because the resulting input would have a different size for every molecule.

A **molecular fingerprint** provides a way around this problem. It converts a molecule of any size into a fixed-length vector of numbers.

One of the most widely used fingerprints is the **Morgan fingerprint**, also known as an **extended-connectivity fingerprint (ECFP)**.


### The basic idea

A Morgan fingerprint describes a molecule by looking at the **local environment around each atom**.

Imagine choosing an atom and asking:

> *What does the chemical neighborhood around this atom look like?*

We can start with just the atom itself and then gradually expand the neighborhood:

* **Radius 0:** the atom itself
* **Radius 1:** the atom and its directly bonded neighbors
* **Radius 2:** everything within two bonds
* **Radius 3:** everything within three bonds

For example, consider the carbon atoms in a molecule:

```text
       O
       |
   C - C - C
       |
       N
```

At radius 0, we only consider the central atom.

At radius 1, we consider the central atom and the atoms directly connected to it.

At radius 2, we include the neighbors of those neighboring atoms as well.

Thus, increasing the radius allows the fingerprint to describe progressively larger pieces of the molecular graph.

### How does this become a vector?

The process can be summarized as follows:

1. **Assign each atom an initial identifier.**
   The identifier is calculated from properties of the atom, such as its element, degree, formal charge, number of attached hydrogens, and whether it belongs to a ring.

2. **Expand the neighborhood.**
   In each round, an atom's identifier is combined with the identifiers of its bonded neighbors. After one round, the identifier describes the atom's environment within one bond; after two rounds, within two bonds; and so on.

3. **Convert each environment into a hash value.**
   Each atom environment is converted into a large integer using a hashing procedure.

4. **Fold the hash values into a fixed-length fingerprint.**
   For a fingerprint with `fpSize = 2048`, each hash value is mapped to one of 2048 positions. The corresponding position is set to `1`.

The result is a vector such as:

```text
[0, 1, 0, 0, 0, 1, 0, ..., 0, 1, 0]
```

The important point is that **every molecule gets the same number of entries**, regardless of how many atoms it contains.

For example:

```text
Molecule A: 10 atoms  →  2048-bit fingerprint
Molecule B: 25 atoms  →  2048-bit fingerprint
Molecule C: 60 atoms  →  2048-bit fingerprint
```

This allows us to stack fingerprints from many molecules into a regular feature matrix that can be given directly to a machine-learning model.

---

### What does a bit mean?

A Morgan fingerprint is usually represented as a **binary vector**.

For each bit:

* `bit[i] = 1` means that **at least one atom environment in the molecule was mapped to that fingerprint position**.
* `bit[i] = 0` means that no atom environment was mapped to that position.

Therefore, a bit is best thought of as indicating the **presence or absence of a molecular substructure or local chemical environment**.

It is **not** a physical measurement, and a value of `1` does not mean that the corresponding feature occurs exactly once.

For example:

```text
Fingerprint:
[0, 1, 0, 0, 1, 0, 0, 1, ...]
    ↑       ↑       ↑
  feature feature feature
```

The fingerprint is therefore a compact description of the chemical environments present in the molecule.

### Why can fingerprints from different molecules be compared?

The same fingerprint generation procedure is applied to every molecule.

Thus, the same hashed environment will be mapped to the same fingerprint position regardless of which molecule it came from.

For example, if a particular chemical environment maps to bit 137:

```text
Molecule A → bit 137 = 1
Molecule B → bit 137 = 0
Molecule C → bit 137 = 1
```

we can compare that feature across molecules.

This is what allows a collection of fingerprints to form a machine-learning feature matrix:

```text
              Fingerprint bits
             0  1  2  3  ... 2047
Molecule 1   0  1  0  0  ...  1
Molecule 2   1  0  0  1  ...  0
Molecule 3   0  1  1  0  ...  0
   ...
```

---

### An important complication: collisions

There are vastly more possible molecular environments than there are positions in a typical fingerprint.

For example, we might use only 2048 bits:

```python
fpSize = 2048
```

Different molecular environments can therefore end up at the same fingerprint position. This is called a **hash collision**.

Consequently, bit 363 should **not** be interpreted as one unique chemical group.

Instead, bit 363 represents a bucket into which one or more possible molecular environments may have been mapped.

Increasing `fpSize` gives the fingerprint more buckets and therefore generally reduces collisions, but it also produces a larger feature matrix.

For this reason, the individual bit numbers themselves are not chemically meaningful. We need additional information from RDKit to determine **which atom environment caused a particular bit to be set**.

That information will be useful in the next section.


---

### 3.1 Fingerprints of a few molecules

Let's generate Morgan fingerprints for some of our complexes and visualize them.

The fingerprints will all have the same length, even though the complexes contain different numbers of atoms.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

mol = Chem.MolFromSmiles("Cn1c(=O)c2c(ncn2C)n(C)c1=O")


def draw_morgan_environment(mol, center_idx, radius):
    """
    Draw molecule with the Morgan environment around
    center_idx highlighted.
    """

    env_bonds = Chem.FindAtomEnvironmentOfRadiusN(
        mol,
        radius,
        center_idx
    )

    atoms = {center_idx}

    for bond_idx in env_bonds:
        bond = mol.GetBondWithIdx(bond_idx)
        atoms.add(bond.GetBeginAtomIdx())
        atoms.add(bond.GetEndAtomIdx())

    return Draw.MolToImage(
        mol,
        size=(400, 300),
        highlightAtoms=list(atoms),
        highlightBonds=list(env_bonds)
    )


for radius in [0, 1, 2]:
    print(f"Radius = {radius}")
    display(draw_morgan_environment(mol, center_idx=0, radius=radius))

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
import matplotlib.pyplot as plt

# A few example molecules
smiles = [
    "CCO",              # ethanol
    "CC(=O)O",          # acetic acid
    "c1ccccc1",         # benzene
    "CC(=O)OC",         # methyl acetate
    "CCN",              # ethylamine
    "CC(C)O",           # isopropanol
]

mols = [Chem.MolFromSmiles(s) for s in smiles]

fps = []

for mol in mols:
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=2,
        nBits=128
    )
    fps.append(np.array(fp))

fps = np.array(fps)

print("Fingerprint matrix shape:", fps.shape)

plt.figure(figsize=(12, 4))

plt.imshow(
    fps,
    aspect="auto",
    cmap="Greys",
    interpolation="nearest"
)

plt.xlabel("Morgan fingerprint bit")
plt.ylabel("Molecule")
plt.title("Different molecules → fixed-length fingerprints")

plt.colorbar(label="Bit value")

plt.show()

### 3.2 Fingerprints of a few complexes from our dataset

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

FP_RADIUS = 3
FP_SIZE = 2048

mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_SIZE)

SEED = 42
N_SHOW = 5

mfp_sample = df.sample(N_SHOW, random_state=SEED)

for _, row in mfp_sample.iterrows():
    fp = mfpgen.GetFingerprint(row['mol'])
    on_bits = list(fp.GetOnBits())
    print(f"{row['csd_code']}  {row['mol'].GetNumAtoms():3d} atoms  "
          f"{len(on_bits):4d} of {FP_SIZE} bits set")
    print(f"   on-bits: {on_bits[:15]} ...")
    print()


In [ ]:
# the same five fingerprints as a picture: black = 1, white = 0
fp_matrix = np.array([list(mfpgen.GetFingerprint(m)) for m in mfp_sample['mol']])

fig, ax = plt.subplots(figsize=(14, 2.5), dpi=150)
ax.imshow(fp_matrix, aspect='auto', cmap='Greys', interpolation='nearest')
ax.set_yticks(range(len(mfp_sample)))
ax.set_yticklabels(mfp_sample['csd_code'], fontsize=12)
ax.set_xlabel('bit index', fontsize=14)
ax.set_title(f'Morgan fingerprints, radius {FP_RADIUS}, {FP_SIZE} bits', fontsize=14)
plt.tight_layout()
plt.show()

print(f'matrix shape: {fp_matrix.shape}')
print(f'fraction of entries set: {fp_matrix.mean():.4f}')


### 3.3 Which substructure set a bit

Section 3.1 said a bit is a hashed substructure, but the fingerprint itself is just 2048
zeros and ones: it does not carry any record of *what* switched each bit on. This cell asks
RDKit to keep that record while it builds the fingerprint, so a bit index can be turned back
into chemistry.

**What is being recorded.** Recall how a Morgan fingerprint is built. For every atom in the
molecule, and for every radius from 0 up to `FP_RADIUS`, RDKit takes the *environment* of
that atom: the set of atoms and bonds within that many bonds of it.

| Radius | Environment of atom $i$ |
|---|---|
| 0 | the atom on its own, with its element, charge and bond count |
| 1 | the atom plus everything directly bonded to it |
| 2 | that, plus everything bonded to those neighbors |
| 3 | one shell further out |

Each environment is hashed to a large integer, and the bit that gets set is that integer
modulo 2048. So one environment sets exactly one bit. A molecule with $N$ atoms produces up
to $4N$ environments, but many of them are repeats and many hash to the same bit, which is
why the printed count of bits set is much smaller than $4N$.

`AdditionalOutput` with `AllocateBitInfoMap` tells the generator to remember, for each bit it
set, the list of `(atom index, radius)` pairs that set it. `GetBitInfoMap()` returns that as
a dictionary: `{bit index: ((atom_idx, radius), (atom_idx, radius), ...)}`.

**Reading the output.** The first line reports how many of the 2048 bits this molecule turned
on; every other bit is 0. Then, for the eight lowest-numbered bits, each `(atom, radius)` pair
that produced it. Three kinds of line show up:

- `bit 38 set by 1 environment(s): atom 67 (Fe) radius 2`
  Bit 38 is on because the region within two bonds of iron atom 67 hashes to 38. Any other
  complex with that same local arrangement around its iron will also have bit 38 on, and
  that is exactly how two fingerprints come to look alike.
- `bit 80 set by 2 environment(s): atom 22 (C) radius 0, atom 52 (C) radius 0`
  Radius 0 means the bare atom, so bit 80 says "this molecule contains a carbon of this
  particular type". Two carbons qualify. The bit is still just 1: a plain fingerprint records
  presence, not how many times.
- `bit 91 set by 9 environment(s):` nine hydrogens, all at radius 1
  Nine hydrogens with identical surroundings, for example three methyl groups. They are the
  same substructure occurring nine times, not nine different substructures.

**Repeats versus collisions.** Both look the same in the vector, but the list tells them
apart. If the atoms sharing a bit are the same element at the same radius in equivalent
positions, they are symmetry-equivalent copies of one substructure, which is honest
behaviour. If they are different elements or different radii, two genuinely unrelated
environments have landed on the same index: that is the hash collision described in 3.1, and
you can now see one directly.

**Why bother.** Bit indices carry no meaning of their own. Bit 363 is not "a pyridine ring";
it is wherever the hash happened to send something. When Part 4 lets a random forest pick the
64 most useful bits, this cell is the only way to ask what those bits were looking at. Change
`demo_idx` to inspect a different complex from the sample of five.


In [ ]:
demo_idx = 0   # 0-4: which of the five sampled complexes to inspect

demo_code = mfp_sample['csd_code'].iloc[demo_idx]
demo_mol = mfp_sample['mol'].iloc[demo_idx]

ao = rdFingerprintGenerator.AdditionalOutput()
ao.AllocateBitInfoMap()
mfpgen.GetFingerprint(demo_mol, additionalOutput=ao)
bit_info = ao.GetBitInfoMap()

print(f'{demo_code}: {len(bit_info)} bits set\n')

for bit in sorted(bit_info)[:8]:
    environments = bit_info[bit]
    print(f'bit {bit:4d}  set by {len(environments)} environment(s):')
    for atom_idx, radius in environments:
        atom = demo_mol.GetAtomWithIdx(atom_idx)
        print(f'    atom {atom_idx:3d} ({atom.GetSymbol():2s})  radius {radius}')


---

## Part 4 - PCA of the fingerprint space

Each complex is now a point in 2048-dimensional binary space. PCA projects that down to two
components so it can be plotted.

Two details in the procedure below are worth naming:

- **Feature selection first.** Most of the 2048 bits say nothing about the redox potential, and
  PCA has no way to know which. A random forest is fitted against `y` and the 64 bits it ranks
  most important are kept; PCA runs on those. The unfiltered projection is computed too
  (`mfp_pca_calculated_full`) if you want to compare.
- **Global scaling, not per-column.** `StandardFlexibleScaler(column_wise=False)` divides the
  whole matrix by one scalar. Scaling each column separately would divide every rare bit by its
  own tiny standard deviation and inflate it to the same weight as a common one.

### 4.1 Helper functions


In [ ]:
# Helper functions for the PCA below. Most of the 2048 fingerprint bits carry nothing
# useful about redox potential, so get_rf_feature_mask fits a random forest against the
# target and keeps only the bits it ranks most important; pca_from_mfp then projects that
# reduced set down to two components. Selecting first keeps the projection from being
# dominated by bits that are irrelevant to the property we care about.
from sklearn.decomposition import PCA
from skmatter.preprocessing import StandardFlexibleScaler
import sklearn.ensemble


def get_rf_feature_mask(x: np.ndarray,
                        y: np.ndarray,
                        n_features: int = None,
                        rf_model: sklearn.ensemble.RandomForestRegressor = None,
                        **rf_params) -> np.ndarray:
    """
    Trains a RandomForestRegressor and returns a boolean mask for the
    top 'n_features' most important features.

    Args:
        x (np.ndarray): Input features (e.g., fingerprints), shape (n_samples, n_original_features).
        y (np.ndarray): Target variable, shape (n_samples,).
        n_features (int, optional): The number of top features to select.
            If None or >= x.shape[1], a mask selecting all features is returned.
        rf_model (RandomForestRegressor, optional): A pre-initialized
            RandomForestRegressor instance. If None, a new one is created.
        **rf_params: Additional keyword arguments passed to the
                     RandomForestRegressor constructor if rf_model is None.

    Returns:
        np.ndarray: A boolean mask of shape (n_original_features,) where True
                    indicates a selected feature.
    """
    n_samples, n_original_features = x.shape

    if n_features is None:
        n_features = n_original_features
        print(f"n_features not specified, selecting all {n_original_features} features.")

    if n_features > n_original_features:
        raise ValueError(f'n_features={n_features} cannot be larger than '
                         f'the original number of features ({n_original_features})')
    elif n_features == n_original_features:
        print(f"n_features ({n_features}) equals original features. Selecting all.")
        return np.ones(n_original_features, dtype=bool)
    else:
        print(f"Selecting top {n_features} features using RandomForestRegressor...")
        if rf_model is None:
            rf_params_default = {'n_estimators': 300, 'n_jobs': -1, 'random_state': 42}
            rf_params_default.update(rf_params)
            print(f"  Creating RandomForestRegressor with params: {rf_params_default}")
            rf = sklearn.ensemble.RandomForestRegressor(**rf_params_default)
        else:
            print("  Using provided RandomForestRegressor model.")
            rf = rf_model

        if y.ndim > 1 and y.shape[1] == 1:
            y = y.ravel()

        rf.fit(x, y)
        importances = rf.feature_importances_

        indices = (-importances).argsort()[:n_features]

        mask = np.zeros(n_original_features, dtype=bool)
        mask[indices] = True
        print(f"  Generated mask selecting {mask.sum()} features.")
        return mask


def get_masked_mfp(mfp_descriptors_full, y, n_features, random_state=42):
    mask = get_rf_feature_mask(mfp_descriptors_full, y, n_features=n_features, random_state=random_state)
    return mfp_descriptors_full[:, mask]


def pca_from_mfp(mfp_descriptors, y=None, n_components=2, n_features=-1, copy=True):
    pca = PCA(n_components=n_components, copy=copy)
    if n_features == -1:
        mfp_pca = pca.fit_transform(mfp_descriptors)
    elif n_features > -1:
        if y is not None:
            mfp_pca = pca.fit_transform(get_masked_mfp(mfp_descriptors, y, n_features))
        else:
            print('Provide target y')
            return None
    else:
        print('n_features must be -1 or above')
        return None
    return mfp_pca


### 4.2 Fingerprinting the whole dataset


In [ ]:
# Build the fingerprint matrix for the whole dataset: one row per complex, 2048 columns.
# Scaling uses a single global factor (column_wise=False) rather than one factor per
# column, because per-column scaling would inflate rare bits that are almost always zero.
# All four PCA variants are kept so the effect of the feature selection can be compared.
def compute_morgan_fingerprint(mol, radius=3, fpSize=2048):
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fpSize)
    morgan_fp = mfpgen.GetFingerprint(mol)   # Get the fingerprint object
    # Convert ExplicitBitVect to NumPy array
    np_fp = np.array(list(morgan_fp))
    return np_fp


descriptors = None

for i, mol in enumerate(df['mol']):
    x = compute_morgan_fingerprint(mol)
    if i == 0:
        descriptors = np.zeros((len(df), x.shape[-1]))
    descriptors[i] = x

mfp_descriptors_scaled_full = StandardFlexibleScaler(column_wise=False).fit_transform(descriptors)

target_y = df['redox_pot'].values

mfp_pca_calculated_full = pca_from_mfp(mfp_descriptors_scaled_full)
mfp_pca_calculated_gnndf_256 = pca_from_mfp(mfp_descriptors_scaled_full, y=target_y, n_features=256)
mfp_pca_calculated_gnndf_128 = pca_from_mfp(mfp_descriptors_scaled_full, y=target_y, n_features=128)
mfp_pca_calculated_gnndf_64 = pca_from_mfp(mfp_descriptors_scaled_full, y=target_y, n_features=64)

print('descriptors:', descriptors.shape)
print('PCA (64 selected bits):', mfp_pca_calculated_gnndf_64.shape)


### 4.3 The projection

Every point is one complex, placed by the first two principal components of its fingerprint and
coloured by its redox potential.


In [ ]:
# The projection, one point per complex, coloured by redox potential. Nothing has been
# clustered yet - this is the plot to look at first and decide by eye whether the
# fingerprint space has any structure in it at all.
pca_2_plot = mfp_pca_calculated_gnndf_256

color_data = df['redox_pot']
color_value_min = -2; color_value_max = 2; colormap = 'coolwarm'
marker_size = 100; marker_alpha = 0.7; marker_edgecolor = 'k'; marker_linewidth = 0.5

fig, ax = plt.subplots(figsize=(8, 6), dpi=100)

xlim = [-0.4, 0.4]; ylim = [-0.4, 0.4]

mappable = ax.scatter(
    x=pca_2_plot[:, 0], y=pca_2_plot[:, 1], c=color_data, cmap=colormap,
    vmin=color_value_min, vmax=color_value_max, s=marker_size,
    alpha=marker_alpha, edgecolor=marker_edgecolor, linewidth=marker_linewidth,
)

ax.set_xlim(xlim); ax.set_ylim(ylim)
ax.xaxis.set_major_locator(ticker.LinearLocator(numticks=5))
ax.yaxis.set_major_locator(ticker.LinearLocator(numticks=5))
ax.set_xlabel("Component 1", fontsize=30)
ax.set_ylabel("Component 2", fontsize=30)
ax.tick_params(axis='both', which='major', labelsize=25)
ax.set_title("PCA", fontsize=30)

cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
cbar = fig.colorbar(mappable, cax=cbar_ax)
cbar.set_label('Redox Potential (V)', fontsize=25)
cbar.ax.yaxis.set_major_locator(ticker.LinearLocator(numticks=5))
cbar.ax.tick_params(labelsize=12)

fig.subplots_adjust(left=0.07, right=0.9, top=0.88, bottom=0.15)
plt.show()


### 4.4 K-Means with k = 2

The same projection, with a two-cluster K-Means fitted to the PCA coordinates. Cluster 1 is
drawn as circles, cluster 2 as crosses; the colour still encodes redox potential.


In [ ]:
# The eye suggested two groups, so ask k-means for exactly two and see whether it agrees.
# Section 1 assigns a cluster label to every complex, section 2 pulls out the reduction
# potentials belonging to each cluster, and section 3 redraws the same PCA plot with a
# different marker shape per cluster so the split sits on top of the colour scale.
from sklearn.cluster import KMeans
from collections import Counter
import copy

df2use = copy.deepcopy(df)

# --- 1. Cluster the PCA Data ---
n_clusters = 2
print(f"Running KMeans clustering on PCA data with k={n_clusters}...")
kmeans = KMeans(n_clusters=n_clusters, n_init='auto', max_iter=1000, tol=0.00001, random_state=52)
cluster_labels = kmeans.fit_predict(pca_2_plot)
df2use['pca_cluster'] = cluster_labels
print(f"Assigned cluster labels (0 to {n_clusters-1}) to {len(df2use)} molecules.")
print("Cluster distribution:", Counter(cluster_labels))

# --- 2. Per-cluster targets ---
color_data_np = df2use['redox_pot'].to_numpy()
cluster_labels_np = df2use['pca_cluster'].to_numpy()

cluster_0_idxs = np.where(cluster_labels_np == 0)[0]
cluster_1_idxs = np.where(cluster_labels_np == 1)[0]

cluster_0_y = df2use.redox_pot[cluster_0_idxs].to_numpy()
cluster_1_y = df2use.redox_pot[cluster_1_idxs].to_numpy()

print(f"Cluster Info:")
print(f"Cluster 0: {len(cluster_0_y)} points, average = {np.mean(cluster_0_y):.2f}, std = {np.std(cluster_0_y):.2f}")
print(f"Cluster 1: {len(cluster_1_y)} points, average = {np.mean(cluster_1_y):.2f}, std = {np.std(cluster_1_y):.2f}")


# --- 3. Plot ---
color_data = df2use['redox_pot']
color_value_min = -2; color_value_max = 2; colormap = 'coolwarm'
marker_size = 100; marker_alpha = 0.7; marker_edgecolor = 'k'; marker_linewidth = 0.5
markers = {0: 'o', 1: 'X'}; cluster_legend_labels = {0: 'Cluster 1', 1: 'Cluster 2'}

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

xlim = [-0.4, 0.4]; ylim = [-0.4, 0.4]

mappable = None
proxy_handles_dict = {}

# Iterate through each cluster to plot with different markers
for cluster_id in range(n_clusters):
    cluster_mask = (cluster_labels == cluster_id)
    if not np.any(cluster_mask):
        continue

    x_cluster = pca_2_plot[cluster_mask, 0]
    y_cluster = pca_2_plot[cluster_mask, 1]
    color_data_cluster = color_data[cluster_mask]

    scatter_plot = ax.scatter(
        x=x_cluster, y=y_cluster, c=color_data_cluster, cmap=colormap,
        vmin=color_value_min, vmax=color_value_max, s=marker_size,
        alpha=marker_alpha, edgecolor=marker_edgecolor,
        linewidth=marker_linewidth, marker=markers[cluster_id],
        label="_nolegend_"
    )
    if cluster_id == 0:
        mappable = scatter_plot
    if cluster_id not in proxy_handles_dict:
        proxy_handles_dict[cluster_id] = ax.scatter(
            [], [], marker=markers[cluster_id], label=cluster_legend_labels[cluster_id],
            edgecolor=marker_edgecolor, linewidth=marker_linewidth + 0.5,
            s=marker_size * 2, c='gray')

ax.set_xlim(xlim); ax.set_ylim(ylim)
ax.xaxis.set_major_locator(ticker.LinearLocator(numticks=5))
ax.yaxis.set_major_locator(ticker.LinearLocator(numticks=5))
ax.set_xlabel("Component 1", fontsize=30)
ax.set_ylabel("Component 2", fontsize=30)
ax.tick_params(axis='both', which='major', labelsize=25)
ax.set_title("PCA", fontsize=30)

sorted_handles = [proxy_handles_dict[cid] for cid in sorted(proxy_handles_dict.keys())]
ax.legend(handles=sorted_handles, loc='lower right', handlelength=0.5,
          handletextpad=0.5, fontsize=25)

cbar_ax = fig.add_axes([0.93, 0.15, 0.02, 0.7])
cbar = fig.colorbar(mappable, cax=cbar_ax)
cbar.set_label('Redox Potential (V)', fontsize=25)
cbar.ax.yaxis.set_major_locator(ticker.LinearLocator(numticks=5))
cbar.ax.tick_params(labelsize=12)

fig.subplots_adjust(left=0.07, right=0.9, top=0.88, bottom=0.15)
plt.show()


### 4.5 Redox potential of the two clusters


In [ ]:
# Do the two clusters differ chemically, or is the split only geometric? Compare the range,
# mean and spread of redox potential in each one before reading anything into them.
print(f"Cluster 1: min={np.min(cluster_0_y):.2f}, max={np.max(cluster_0_y):.2f}, mean={np.mean(cluster_0_y):.2f}, std={np.std(cluster_0_y):.2f}")
print(f"Cluster 2: min={np.min(cluster_1_y):.2f}, max={np.max(cluster_1_y):.2f}, mean={np.mean(cluster_1_y):.2f}, std={np.std(cluster_1_y):.2f}")


In [ ]:
# The same comparison as a picture. Summary statistics hide the shape of a distribution,
# so overlay the two histograms to see how much the clusters overlap and where they part.
plt.figure(figsize=(10, 6), dpi=150)

ax = sns.histplot(cluster_0_y, kde=True, bins=25, label='PCA cluster 1', color='salmon', alpha=0.9, hatch='//')
sns.histplot(cluster_1_y, kde=True, bins=25, label='PCA cluster 2', color='skyblue', alpha=0.7, ax=ax)

plt.xlabel('Redox Potential (V)', fontsize=35)
plt.xlim([-2, 3])
plt.xticks(fontsize=25)

plt.ylabel('Count', fontsize=35)
plt.ylim([0, 600])
plt.yticks(fontsize=25)

plt.legend(fontsize=25)

ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
ax.xaxis.set_minor_locator(AutoMinorLocator(4))
ax.tick_params(which='major', length=8, width=2)
ax.tick_params(axis='x', which='minor', length=5, width=1.5)

plt.tight_layout()
plt.show()


---

## Part 5 - Ligand classification

The clusters came out of fingerprint space, which means they were separated by substructures.
To find out which substructures, break each complex into its ligands and sort those ligands
into chemical classes.

### Why bother

A bit index is not a chemical explanation. Part 4 showed that the dataset splits in two and
that the two halves have different redox potentials, but "the split is driven by bit 363" is
not something you can reason about or design against. Ligands are the natural chemical unit
of a coordination complex, so re-describing each complex by the *kinds* of ligand it carries
turns the split into a statement a chemist can act on.

It is also the quantity that matters physically. The iron centre is the same in every
complex here; what changes the redox potential is the ligand field around it: whether
the donors are pi-acceptors or sigma-donors, how much charge they carry, how strongly they
bind. Ligand class is a coarse proxy for that.

Finally it is a matter of scale. The dataset contains over a thousand distinct ligands,
which is far too many to look at one by one, but they collapse into about ten classes, few
enough to put on a single pie chart and read at a glance. That same chart doubles as a
census of the dataset, which is worth having before training anything on it.

Note that these classes are defined by structure (aromaticity, rings, which heteroatoms are
present) and not directly by donor strength or charge, so they are a starting point for
interpretation rather than the electronic answer itself.

### How a ligand is classified

Deleting the iron atom leaves the ligands as disconnected fragments, and each fragment is
converted to a canonical SMILES so the same ligand appearing in different complexes is
recognised as the same string. `classify_smiles` then applies a rule cascade, stopping at
the first rule that matches:

| # | Class | Rule | Example |
|---|---|---|---|
| 1 | `Inorganic` | on the pre-built inorganic list (no C-C or C-H bonds) | `N` (ammonia) |
| 2 | `Halogen` | a bare halide ion | `[Cl-]` (chloride) |
| 3 | `Acyclic_CH` | not aromatic, no ring, carbon and hydrogen only | `[CH3-]` (methyl) |
| 4 | `Acyclic_Hetero` | not aromatic, no ring, at least one heteroatom | `CO` (methanol) |
| 5 | `Alicyclic_CH` | not aromatic, has a ring, carbon and hydrogen only | `[CH-]1CC1` (cyclopropyl) |
| 6 | `Alicyclic_Hetero` | not aromatic, has a ring, at least one heteroatom | `C1CCOC1` (tetrahydrofuran) |
| 7 | `Aromatic_CH` | aromatic, no heteroatoms | `[c-]1ccccc1` (phenyl) |
| 8 | `Aromatic_monoHetero` | aromatic, one heteroatom element, and it is not N | `[c-]1ccco1` (furanyl) |
| 9 | `Aromatic_CHN(O)` | aromatic, heteroatoms are N, or N and O | `c1ccncc1` (pyridine) |
| 10 | `Aromatic_multiHetero` | aromatic, any other heteroatom combination | `Clc1cnccn1` (chloropyrazine) |

Rows 7-10 count *distinct heteroatom elements*, not heteroatom atoms. Pyrimidine has two
nitrogens but only one heteroatom element, so it lands in `Aromatic_CHN(O)`; chloropyrazine
has N and Cl, which is two elements but not the N/O pair, so it falls through to
`Aromatic_multiHetero`.

### 5.1 Splitting off the ligands and classifying them


In [ ]:
# The three functions that turn a complex into a list of classified ligands.
# get_ligand_smiles_after_fe_removal deletes the iron atom and returns each remaining
# fragment as a canonical SMILES, so the same ligand always produces the same string no
# matter which complex it came from. classify_smiles applies the rule cascade above.
def get_ligand_smiles_after_fe_removal(mol):
    """
    Finds and removes the first 'Fe' atom from an RDKit molecule
    and returns the SMILES strings of the resulting fragments.

    Args:
        mol: An RDKit molecule object.

    Returns:
        A list of SMILES strings for the fragments remaining after
        removing the 'Fe' atom. Returns an empty list if no 'Fe'
        atom is found or if no fragments remain.
    """
    if not mol:
        print("Warning: Received an invalid molecule object.")
        return []

    fe_index = -1
    # Find the index of the first Fe atom
    for atom in mol.GetAtoms():
        if atom.GetSymbol() == 'Fe':
            fe_index = atom.GetIdx()
            break   # Stop after finding the first one

    if fe_index == -1:
        # If no Fe, the "fragments after removal" is an empty concept
        return []

    # Create an editable copy of the molecule
    rwmol = Chem.RWMol(mol)

    # Remove the Fe atom using its index
    rwmol.RemoveAtom(fe_index)

    # Convert back to a standard Mol object
    mol_without_fe = rwmol.GetMol()

    if not mol_without_fe:
        # This might happen if the original molecule was just [Fe]
        return []

    # Get the disconnected fragments as separate Mol objects
    # sanitizeFrags=False prevents errors from valences left behind by the Fe removal
    fragments = Chem.GetMolFrags(mol_without_fe, asMols=True, sanitizeFrags=False)

    # Convert each fragment molecule to SMILES
    fragment_smiles = [Chem.MolToSmiles(Chem.MolFromSmiles(Chem.MolToSmiles(frag))) for frag in fragments]

    return fragment_smiles


# RDKit to check if *any* atom is aromatic.
def is_molecule_aromatic(mol):
    """Checks if any atom in the molecule is aromatic."""
    if not mol:
        return False
    for atom in mol.GetAtoms():
        if atom.GetIsAromatic():
            return True
    return False


def classify_smiles(smiles_string, inorganic_list, halogen_list):
    """
    Classifies a SMILES string based on the user's defined rules.

    Args:
        smiles_string (str): The input SMILES string.
        inorganic_list (list or set): List/set of SMILES considered 'inorganic'.
        halogen_list (list or set): List/set of SMILES considered elemental 'halogen'.

    Returns:
        str: The classification category.
    """
    # --- Predefined Sets (Atomic Numbers) ---
    HALOGENS_ATNUM = {9, 17, 35, 53, 85}   # F, Cl, Br, I, At
    TARGET_HETERO_ATNUM = {7,}   # N,
    OTHER_HETERO_ATNUM = {16, 15, 14, 34, 33, 51, 52, 81, 82, 83, 84}   # S, P, Si, Se, As, Sb, Te, Tl, Pb, Bi, Po
    OXYGEN_ATNUM = {8}
    CARBON_ATNUM = {6}
    ALLOWED_HOMO_ATNUM = {6, 1}   # Carbon, Hydrogen (for atom iteration)

    # --- Initial Checks ---
    if not isinstance(smiles_string, str) or not smiles_string:
        return "invalid_input"

    # Check predefined lists first
    if smiles_string in inorganic_list:
        return "Inorganic"
    if smiles_string in halogen_list:
        return "Halogen"   # Covers single halogen atoms like 'Cl' if in list

    # --- RDKit Molecule Processing ---
    mol = Chem.MolFromSmiles(smiles_string)
    if mol is None:
        # Try sanitizing, might help with some unusual SMILES
        mol = Chem.MolFromSmiles(smiles_string, sanitize=True)
        if mol is None:
            return "invalid_smiles"   # Cannot parse SMILES

    n_atoms = mol.GetNumHeavyAtoms()   # Get number of non-hydrogen atoms

    # --- Classification Logic ---

    # 1. Single Heavy Atom Check (after list checks)
    if n_atoms == 1:
        atom = mol.GetAtomWithIdx(0)
        atom_symbol = atom.GetSymbol()
        if atom_symbol in halogen_list:
            return "Halogen"
        else:
            mol = Chem.AddHs(mol)

    # 2. Determine Core Properties
    is_aromatic = is_molecule_aromatic(mol)
    has_halogen = any(atom.GetAtomicNum() in HALOGENS_ATNUM for atom in mol.GetAtoms())
    num_rings = mol.GetRingInfo().NumRings()

    # --- Aliphatic Classification ---
    if not is_aromatic:
        has_hetero = any(
            atom.GetAtomicNum() not in CARBON_ATNUM | {1}   # Exclude C, H
            for atom in mol.GetAtoms() if atom.GetAtomicNum() != 1   # Iterate heavy atoms
        )

        if num_rings == 0:   # Acyclic
            if has_hetero:
                return "Acyclic_Hetero"
            else:
                return "Acyclic_CH"   # Only C, H (Halogens already excluded)

        else:   # Alicyclic, num_rings > 0
            if has_hetero:
                return "Alicyclic_Hetero"
            else:
                return "Alicyclic_CH"   # Only C, H (Halogens already excluded)

    # --- Aromatic Classification ---
    else:   # is_aromatic True
        heavy_atoms = [atom for atom in mol.GetAtoms() if atom.GetAtomicNum() != 1]
        present_atomic_nums = {atom.GetAtomicNum() for atom in heavy_atoms}

        all_hetero_ATnums = present_atomic_nums - CARBON_ATNUM - {1}   # Heteroatoms (excluding C, H)
        num_heteroatoms = len(all_hetero_ATnums)

        if num_heteroatoms == 0:   # no hetero atoms
            return 'Aromatic_CH'
        elif num_heteroatoms == 1:
            if {7}.issubset(all_hetero_ATnums):   # contains N
                return 'Aromatic_CHN(O)'
            else:
                return 'Aromatic_monoHetero'
        elif num_heteroatoms == 2:
            if {7, 8}.issubset(all_hetero_ATnums):   # contains N and O
                return 'Aromatic_CHN(O)'
            else:
                return 'Aromatic_multiHetero'
        elif num_heteroatoms > 2:
            return 'Aromatic_multiHetero'

    # Fallback if somehow no category is matched (should not happen with this logic)
    return "unknown"


In [ ]:
# Run every complex through those functions. The unique ligands are collected first, then
# the inorganic ones are identified (thermo's is_organic, minus anything holding a C-C or
# C-H bond, minus bare halides) because classify_smiles needs that list to test rule 1.
# The result is classes_info: class name -> the unique ligands belonging to it.
import operator
from collections import defaultdict

from thermo.functional_groups import *

for i, mol in enumerate(df2use['mol'][:]):
    if i == 0:
        fragment_smiles_list = []
    fragment_smiles = get_ligand_smiles_after_fe_removal(mol)
    fragment_smiles_list.append(fragment_smiles)

unique_ligands = set()
for i, smiles_list in enumerate(fragment_smiles_list):
    for smiles in smiles_list:
        unique_ligands.add(smiles)
unique_ligands = list(unique_ligands)


halogens = ['F', 'Cl', 'Br', 'I']
exclude_smiles = []
inorganic_ligands = []
for smiles in unique_ligands:
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    if not is_organic(mol):
        # check if there are any C-C bonds or C-H bonds
        for bond in mol.GetBonds():
            if bond.GetBondTypeAsDouble() == 1.0:
                # check if both atoms are carbon
                atom1 = bond.GetBeginAtom()
                atom2 = bond.GetEndAtom()
                bond_sign = set([atom1.GetSymbol(), atom2.GetSymbol()])
                if bond_sign == {'C', 'C'} or bond_sign == {'C', 'H'}:
                    exclude_smiles.append(smiles)
        # Single halogen atoms are also considered inorganic
        if len(mol.GetAtoms()) == 1 and mol.GetAtoms()[0].GetSymbol() in halogens:
            exclude_smiles.append(smiles)
        if smiles not in exclude_smiles:
            inorganic_ligands.append(smiles)


classes_info = defaultdict(list)
ligand_cls_info = {}

for mol in df2use['mol']:
    lig_smiles = get_ligand_smiles_after_fe_removal(mol)
    for smiles in lig_smiles:
        classification = classify_smiles(smiles, inorganic_ligands, halogens)
        if smiles not in classes_info[classification]:
            classes_info[classification].append(smiles)
        if smiles not in ligand_cls_info:
            ligand_cls_info[smiles] = classification
        else:
            # If already present, check if classifications match
            if ligand_cls_info[smiles] != classification:
                print(f"  Conflict for SMILES {smiles}: {ligand_cls_info[smiles]} vs {classification}")

print(f'{len(unique_ligands)} unique ligands, {len(inorganic_ligands)} of them inorganic')
for class_name, smiles_list in classes_info.items():
    print(f'  {class_name:22s} {len(smiles_list):4d} unique ligands')


### 5.2 How common is each class


In [ ]:
# How common each class is across the dataset. This counts ligand *occurrences*, not unique
# ligands, so a ligand appearing in 200 complexes contributes 200 - that measures what the
# dataset is made of, rather than how large its ligand vocabulary is.
from matplotlib import colormaps
import matplotlib.patheffects as path_effects

all_ligands = []
all_ligand_cls_counter = defaultdict(int)
for mol in df2use['mol']:
    lig_smiles = get_ligand_smiles_after_fe_removal(mol)
    all_ligands.extend(lig_smiles)
    for smiles in lig_smiles:
        classification = classify_smiles(smiles, inorganic_ligands, halogens)
        all_ligand_cls_counter[classification] += 1
all_ligand_cls_freq = {label: (count / len(all_ligands)) * 100 for label, count in all_ligand_cls_counter.items()}
sorted_ligand_freqs = sorted(all_ligand_cls_freq.items(), key=operator.itemgetter(1), reverse=True)


total_ligands = sum(all_ligand_cls_counter.values())


# Calculate percentages
all_ligand_cls_percent = {
    label: (count / total_ligands) * 100
    for label, count in all_ligand_cls_counter.items()
}


# --- Custom Sorting Function ---
def sort_key(item):
    label, perc = item
    if label.lower().startswith('aro'): group = 0
    elif label.lower().startswith('ali'): group = 1
    elif label.lower().startswith('acy'): group = 1
    else: group = 2
    return (group, -perc)   # Sort by group, then by percentage descending


# Sort items
sorted_items = sorted(all_ligand_cls_percent.items(), key=sort_key)

# Separate labels, percentages, and counts
sorted_labels = [item[0] for item in sorted_items]
sorted_percents = [item[1] for item in sorted_items]
sorted_counts = [all_ligand_cls_counter[label] for label in sorted_labels]

# --- Pie Chart Plotting ---

fig, ax = plt.subplots(figsize=(12, 10), dpi=150)

# Define threshold
percent_threshold = 2.0


# Custom autopct function (only show >= threshold)
def func_autopct(pct):
    return f"{pct:.1f}%" if pct >= percent_threshold else ""


# --- Get the colormap ---
n_slices = len(sorted_counts)
cmap = colormaps.get_cmap('Spectral')
slice_colors = cmap(np.linspace(1, 0, n_slices))


# --- Plot the pie chart WITH SPECIFIED COLORS ---
radius = 0.7
wedges, texts, autotexts = ax.pie(
    sorted_counts,
    autopct=func_autopct,
    startangle=210,
    pctdistance=0.7,   # Place percentage inside
    colors=slice_colors,
    wedgeprops={'linewidth': 0.5, 'edgecolor': 'white'},
    radius=radius
)

# Style autopct text
for autotext in autotexts:
    autotext.set_color('k')
    autotext.set_weight('bold')
    autotext.set_fontsize(18)


# --- Custom Labeling and Legend Preparation ---
# Base settings for annotate
kw = dict(arrowprops=dict(arrowstyle="-", color='gray', lw=0.7),
          zorder=0, va="center")

# Fixed distance for large slice labels
large_slice_label_radius = 1.1 * radius

# Lists to store items for the legend
legend_handles = []
legend_labels = []

for i, p in enumerate(wedges):
    # Get data for this wedge
    percentage = sorted_percents[i]
    label_text = sorted_labels[i]
    wedge_color = p.get_facecolor()

    if percentage >= percent_threshold:
        # --- Large Slices: Annotate beside the pie ---
        ang = (p.theta2 - p.theta1) / 2. + p.theta1   # Angle of the middle
        y = np.sin(np.deg2rad(ang))
        x = np.cos(np.deg2rad(ang))

        # Use a straight line connector for simplicity
        kw['arrowprops'].update({"connectionstyle": None, "arrowstyle": "-", "color": 'k', "lw": 1})

        # Horizontal alignment
        horizontalalignment = {-1: "right", 1: "left"}[int(np.sign(x))]

        # Calculate text position
        text_x = large_slice_label_radius * x
        text_y = large_slice_label_radius * y

        # --- Draw annotation for large slices ---
        ax.annotate(label_text,
                    xy=(x * 0.98 * radius, y * 0.98 * radius),   # Point on wedge edge
                    xytext=(text_x, text_y),
                    horizontalalignment=horizontalalignment,
                    color='k',
                    fontsize=22,
                    fontweight='bold',
                    path_effects=[
                        path_effects.Stroke(linewidth=1.5, foreground=wedge_color),
                        path_effects.Normal()],
                    **kw)
    else:
        # --- Small Slices: Add to legend ---
        legend_handles.append(plt.Rectangle((0, 0), 1, 1, fc=wedge_color))
        legend_labels.append(f"{label_text} ({percentage:.1f}%)")


# --- Add the Legend for Small Slices ---
if legend_handles:
    leg = ax.legend(legend_handles, legend_labels,
                    title="Other Classes (<{:.1f}%)".format(percent_threshold),
                    loc="center left",
                    bbox_to_anchor=(-0.05, 0.95),
                    ncol=1,
                    fontsize=20,
                    title_fontsize=20,
                    frameon=False)

    # Set legend text colors to match pie slices (handles)
    for i, text in enumerate(leg.get_texts()):
        text.set_color('k')
        text.set_path_effects([
            path_effects.Stroke(linewidth=0.55, foreground=wedge_color),
            path_effects.Normal()
        ])

plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()

print(f'{total_ligands} ligands in total across {len(df2use)} complexes')


### 5.3 Example ligands from each class

Up to five ligands drawn at random from every class, so the class names above can be checked
against what the molecules actually look like.


In [ ]:
# What the classes actually look like: a grid of example ligands drawn from each one.
# Reading the structures is the check on whether the rule cascade is doing something
# chemically sensible. random.sample is unseeded, so the examples change on every run.
import random
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display, Image as IPImage   # Import IPImage for displaying PIL images
from PIL import Image, ImageDraw, ImageFont   # Import Pillow modules


for class_name, smiles_list in classes_info.items():
    print(f"--- Class: {class_name} ---")

    if not smiles_list:
        print(f"  No molecules in class '{class_name}' to display.")
        print("\n" + "="*50 + "\n")
        continue

    # Determine the number of molecules to sample (up to 5)
    num_to_sample = min(5, len(smiles_list))

    if num_to_sample == 0:
        print(f"  No molecules to sample in class '{class_name}'.")
        print("\n" + "="*50 + "\n")
        continue

    selected_smiles = random.sample(smiles_list, num_to_sample)

    mols_to_draw = []
    legends_for_mols = []
    for smi in selected_smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mols_to_draw.append(mol)
            legends_for_mols.append(smi)
        else:
            print(f"  Warning: Could not parse SMILES '{smi}' from class '{class_name}'. Skipping.")

    if not mols_to_draw:
        print(f"  No valid molecules to draw for class '{class_name}'.")
        print("\n" + "="*50 + "\n")
        continue

    # --- Legend Text Size Adjustment ---
    # This value impacts the overall size of each molecule drawing.
    sub_image_width = 500
    sub_image_height = 500

    # Create the grid image (as a PIL Image object)
    draw_opts = Draw.rdMolDraw2D.MolDrawOptions()
    draw_opts.legendFontSize = 30      # or whatever size you like

    pil_image = Draw.MolsToGridImage(
        mols_to_draw,
        molsPerRow=5,
        subImgSize=(sub_image_width, sub_image_height),
        legends=legends_for_mols,
        drawOptions=draw_opts,
        returnPNG=False
    )

    # --- Add Title/Text to the PIL Image ---
    if pil_image:
        # Make the image editable
        drawable_image = pil_image.convert("RGBA")
        draw = ImageDraw.Draw(drawable_image)

        # --- Font for the title ---
        font_size = 60
        try:
            font = ImageFont.truetype("arial.ttf", font_size)
        except IOError:
            try:
                # DejaVuSans ships with matplotlib, so it is there on Linux too
                font = ImageFont.truetype("DejaVuSans.ttf", font_size)
            except IOError:
                print("  Warning: no TrueType font found. Using default PIL font for title.")
                font = ImageFont.load_default()
                font_size = 20

        # --- Position and draw the title ---
        title_text = str(class_name)
        try:
            text_bbox = font.getbbox(title_text)   # For Pillow >= 9.2.0
            text_width = text_bbox[2] - text_bbox[0]
            text_height = text_bbox[3] - text_bbox[1]
        except AttributeError:   # Fallback for older Pillow
            try:
                text_width, text_height = draw.textsize(title_text, font=font)
            except AttributeError:
                text_width, text_height = (len(title_text) * font_size / 2, font_size)

        padding = 10
        text_x = padding
        text_y = padding

        draw.text((text_x, text_y), title_text, font=font, fill=(0, 0, 0, 255))

        # --- Display the modified image ---
        import io
        img_byte_arr = io.BytesIO()
        drawable_image.save(img_byte_arr, format='PNG')
        img_byte_arr = img_byte_arr.getvalue()
        display(IPImage(data=img_byte_arr))

    else:
        print(f"  Failed to generate grid image for class '{class_name}'.")

    print("\n" + "="*50 + "\n")


### 5.4 Iron coordination number

How many donor atoms surround the metal, counted straight off the molecular graph.


In [ ]:
# Coordination number = how many bonds the iron makes. GetDegree() returns the number of
# bonded neighbours of an atom, which for these reconstructed complexes is the number of
# donor atoms around the metal. The histogram shows how much of the dataset is octahedral.

# List to store the results
coordination_numbers = []
skipped = 0


for idx, row in df.iterrows():
    mol = row['mol']

    if mol is None:
        skipped += 1
        continue

    # Find the Iron Atom (Z=26)
    fe_atom = None
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 26:
            fe_atom = atom
            break

    if fe_atom:
        # GetDegree() returns the number of bonds (neighbors) the atom has
        cn = fe_atom.GetDegree()
        coordination_numbers.append(cn)
    else:
        # No Iron found in this mol
        skipped += 1

print(f"Skipped {skipped} (No Mol or No Iron found).")

# Convert to Series for easier stats
cn_series = pd.Series(coordination_numbers)
print("\n--- Distribution Stats ---")
print(cn_series.value_counts().sort_index())


plt.figure(figsize=(8, 6))

# Use Seaborn with discrete=True to center bars on integers
sns.histplot(coordination_numbers, discrete=True, color='steelblue', shrink=0.8)

# Calculate percentage for annotations
total_counts = len(coordination_numbers)
counts = cn_series.value_counts().sort_index()

# Annotate bars with counts/percentages
ax = plt.gca()
for cn, count in counts.items():
    pct = (count / total_counts) * 100
    scale = 5
    padding = 30 if cn % 2 == 1 else 10
    # Place text slightly above the bar
    ax.text(cn, count+padding, f"{pct:.1f}%", ha='center', va='bottom', fontsize=16, fontweight='bold')

# Labels and Styling
#plt.title('Distribution of Iron Coordination Numbers', fontsize=14)
plt.xlabel('Fe Coordination Number', fontsize=25)
plt.ylabel('Count', fontsize=25)

plt.ylim([0,1200])

# Force X-axis to show only relevant integers (e.g., 4, 5, 6)
unique_cns = sorted(list(set(coordination_numbers)))
plt.xticks(unique_cns)

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()



**Mentor checkpoint 7**: after the structural analysis (Parts 2-5)

- What does a set bit in a Morgan fingerprint mean, and why can two unrelated substructures
  end up setting the same bit?
- The PCA projection used only the 64 bits a random forest ranked highest. What does that
  choice do to the picture, and what would change if all 2048 bits were kept?
- The two K-Means clusters differ in mean redox potential. Is that a chemical result, or a
  consequence of how the fingerprint space was built and projected?
- The ligand classes are defined by structure, not by donor strength or charge. Which
  classes would you expect to move the redox potential most, and how would you test it?

Proceed only after confirmation.

---


---

## Part 6 - Graphs and molecular graphs

A great many objects in science can be described as a **graph**: a set of things together
with the relationships between them. People connected by friendships, cities by roads,
airports by flights, web pages by hyperlinks, and the case that matters here, atoms
connected by chemical bonds.

### 6.1 What is a graph?

A graph has two ingredients, **nodes** (also called vertices) and **edges** (also called
links), and is written

$$
G = (V, E)
$$

where $V$ is the set of nodes and $E$ the set of edges. For example:

```text
       B
      / \
     A   D
      \ /
       C
```

Here $V = \{A, B, C, D\}$ and $E = \{(A,B), (A,C), (B,D), (C,D)\}$. The nodes are the
objects, the edges are the relationships between them, and what each one means depends
entirely on the system being described:

| System | Nodes | Edges |
|---|---|---|
| Social network | People | Friendships |
| Road network | Intersections | Roads |
| Airline network | Airports | Flights |
| Citation network | Papers | Citations |
| Molecule | Atoms | Chemical bonds |

The same mathematics covers all of them. Here is that four-node graph in NetworkX.


In [ ]:
# A graph in its simplest form: a set of nodes and a set of edges between them.
# spring_layout only decides where to draw the nodes; it has no effect on the graph itself.

G = nx.Graph()
G.add_nodes_from(['A', 'B', 'C', 'D'])
G.add_edges_from([('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D')])

plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_size=1200, font_size=14, width=2)
plt.title('A graph: nodes and edges')
plt.axis('off')
plt.show()


### 6.2 Graphs can carry data

A graph is more than dots and lines. **Features** can be attached to the nodes and to the
edges. If this graph were a social network, each person might carry an age.


In [ ]:
# The same four nodes, now each carrying one feature. The colour encodes the value, so the
# picture shows both things at once: who is connected to whom, and what each node holds.

ages = {'A': 22, 'B': 35, 'C': 28, 'D': 41}

plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, node_size=1800, node_color=list(ages.values()), cmap='viridis', width=2)

nx.draw_networkx_labels(
    G, pos,
    labels={node: f'{node}\nAge: {age}' for node, age in ages.items()},
    font_size=10,
)

plt.colorbar(
    plt.cm.ScalarMappable(cmap='viridis',
                          norm=plt.Normalize(min(ages.values()), max(ages.values()))),
    ax=plt.gca(), label='Age',
)
plt.title('A graph with node features')
plt.axis('off')
plt.show()


The **structure** says who is connected to whom; the **node features** say something about
each node individually. Keeping those two apart is what makes the molecular version useful.


### 6.3 A molecule is a graph

Ethanol, $\mathrm{CH_3-CH_2-OH}$, written as a graph:

```text
C ----- C ----- O
o       o       o
```

The atoms become nodes and the bonds become edges.

> **A molecular graph represents atoms as nodes and chemical bonds as edges.**

This is a different choice from Part 3. A Morgan fingerprint flattens a molecule into a
fixed 2048-bit vector by hashing its substructures, and the connectivity is gone once the
bits are set. A graph keeps the connectivity and hands it to the model directly.

For machine learning a molecular graph carries information at three levels.

**Node features.** Each atom gets a feature vector: atomic number, formal charge, number of
neighbors, attached hydrogens, hybridization, aromaticity, ring membership. For $N$ atoms
and $F$ features per atom these stack into a matrix

$$
X =
\begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1F} \\
x_{21} & x_{22} & \cdots & x_{2F} \\
\vdots & \vdots & \ddots & \vdots \\
x_{N1} & x_{N2} & \cdots & x_{NF}
\end{bmatrix}
$$

with one row per atom and one column per feature.

**Edge features.** Each bond can carry its own vector too: single, double or triple,
aromatic, conjugated, in a ring.

**Graph-level properties.** The quantity attached to the molecule as a whole, which is
usually what we want to predict: energy, dipole moment, HOMO/LUMO gap, solubility, or in
this project the redox potential. Because the target belongs to the entire graph rather
than to any one atom, this is called a **graph-level prediction**.

| Level | Molecular meaning | Example |
|---|---|---|
| Node | Atom | C, N, O, Fe |
| Edge | Bond | Single, double, aromatic |
| Graph | Whole molecule | Redox potential |


### 6.4 Building a molecular graph with RDKit

An RDKit molecule already is a graph. It can be asked for its atoms, which are the nodes,
and for its bonds, which are the edges.


In [ ]:
# SMILES in, molecule out. Everything below reads the graph off this object.

smiles = 'CCO'
mol = Chem.MolFromSmiles(smiles)

display(Draw.MolToImage(mol, size=(500, 300)))


In [ ]:
# The nodes. Each atom carries an index, and that index is the node label.

for atom in mol.GetAtoms():
    print(atom.GetIdx(), atom.GetSymbol())


In [ ]:
# The edges. Each bond is a pair of atom indices, which together form the edge list.

for bond in mol.GetBonds():
    print(bond.GetBeginAtomIdx(), '--', bond.GetEndAtomIdx())


So ethanol is three nodes and two edges:

```text
   0       1       2
   C ----- C ----- O
```

Nodes `0, 1, 2` with edge list `(0, 1)` and `(1, 2)`. Together these define the
connectivity of the graph.


### 6.5 The same graph in NetworkX


In [ ]:
# Copy the molecule into a NetworkX graph: atoms become nodes (keeping the element symbol
# as a node attribute), bonds become edges. Nothing chemical is added here, this is the same
# information in a general-purpose container.

G_mol = nx.Graph()

for atom in mol.GetAtoms():
    G_mol.add_node(atom.GetIdx(), element=atom.GetSymbol())

for bond in mol.GetBonds():
    G_mol.add_edge(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())

plt.figure(figsize=(7, 5))
pos = nx.spring_layout(G_mol, seed=42)
nx.draw_networkx_edges(G_mol, pos, width=3)
nx.draw_networkx_nodes(G_mol, pos, node_size=1800)
nx.draw_networkx_labels(
    G_mol, pos,
    labels={n: f"{n}\n{G_mol.nodes[n]['element']}" for n in G_mol.nodes},
    font_size=11,
)
plt.title('Ethanol as a graph')
plt.axis('off')
plt.show()


The drawing does not look like the usual chemical structure, and that is fine. A graph
records **connectivity**, not geometry. Where the nodes land on the page is a decision made
by the layout algorithm and carries no chemical meaning.


### 6.6 Node features

RDKit will report the per-atom properties that make up the rows of $X$.


In [ ]:
# One row per atom, one column per feature: this table is exactly the node feature matrix X
# from 6.3, before it is converted to numbers a model can consume.

node_data = []
for atom in mol.GetAtoms():
    node_data.append({
        'Atom index': atom.GetIdx(),
        'Element': atom.GetSymbol(),
        'Atomic number': atom.GetAtomicNum(),
        'Degree': atom.GetDegree(),
        'Formal charge': atom.GetFormalCharge(),
        'H count': atom.GetTotalNumHs(),
        'Aromatic': atom.GetIsAromatic(),
    })

node_df = pd.DataFrame(node_data)
display(node_df)


### 6.7 Adjacency matrix

Connectivity can also be written as a square matrix with one row and one column per node.
Entry $A_{ij}$ is 1 when nodes $i$ and $j$ are bonded and 0 when they are not. For the
four-node graph of 6.1:

```text
      A B C D
    +--------
A   | 0 1 1 0
B   | 1 0 0 1
C   | 1 0 0 1
D   | 0 1 1 0
```

and for ethanol,

$$
A =
\begin{bmatrix}
0 & 1 & 0 \\
1 & 0 & 1 \\
0 & 1 & 0
\end{bmatrix}
$$


In [ ]:
# The adjacency matrix of the ethanol graph, printed and drawn. It is symmetric because the
# bonds have no direction, and the diagonal is zero because no atom is bonded to itself.

A = nx.to_numpy_array(G_mol, dtype=int)
print(A)

n_atoms = mol.GetNumAtoms()

plt.figure(figsize=(5, 4))
plt.imshow(A, interpolation='nearest')
plt.xticks(range(n_atoms), range(n_atoms))
plt.yticks(range(n_atoms), range(n_atoms))
plt.xlabel('Atom')
plt.ylabel('Atom')
plt.title('Molecular graph: adjacency matrix')
plt.colorbar(label='Connected')
plt.show()


The chemical drawing and the adjacency matrix hold the same connectivity. One is a visual
representation, the other a numerical one, and only the second can be fed to a model.


### 6.8 Degree and neighborhoods

The **degree** of a node is the number of edges attached to it, which for a molecule is the
number of directly bonded atoms. This is the same quantity counted for iron in 5.4.


In [ ]:
# Degree of every atom in ethanol: the two end atoms have one bond each, the middle carbon
# has two. Colour encodes the degree so it can be read off the drawing.

degrees = dict(G_mol.degree())
print(degrees)

plt.figure(figsize=(7, 5))
pos = nx.spring_layout(G_mol, seed=42)
nx.draw(G_mol, pos, node_size=1800, node_color=list(degrees.values()), width=3)
nx.draw_networkx_labels(
    G_mol, pos,
    labels={n: f"{G_mol.nodes[n]['element']}\nDegree={d}" for n, d in degrees.items()},
    font_size=10, font_color='red',
)
plt.title('Node degree')
plt.axis('off')
plt.show()


The **neighborhood** of a node is the set of nodes one edge away from it. Take the middle
carbon of ethanol:

```text
C ----- C ----- O
        ^
      center
```

Its neighborhood is the carbon on one side and the oxygen on the other.


In [ ]:
# The neighbourhood of atom 1: everything one bond away. The colours mark the centre, its
# neighbours, and the rest of the graph.

center = 1
neighbors = list(G_mol.neighbors(center))
print('Center atom:', center)
print('Neighbors: ', neighbors)

node_colors = [2 if n == center else (1 if n in neighbors else 0) for n in G_mol.nodes]

plt.figure(figsize=(7, 5))
pos = nx.spring_layout(G_mol, seed=42)
nx.draw(G_mol, pos, node_size=1800, node_color=node_colors, width=3)
nx.draw_networkx_labels(
    G_mol, pos,
    labels={n: f"{n}\n{G_mol.nodes[n]['element']}" for n in G_mol.nodes},
    font_size=10, font_color='red',
)
plt.title('Neighborhood of atom 1')
plt.axis('off')
plt.show()


Read the other way round, the central atom is **receiving information** from the atoms
bonded to it:

```text
        C  -->  [C]  <--  O
                 |
               center
```

That single idea is the foundation of message passing in a graph neural network.


### 6.9 A larger graph: caffeine

Nothing changes for a bigger molecule except the number of nodes and edges.


In [ ]:
# Same two loops as 6.5, applied to caffeine. Node colour encodes the element, so the two
# fused rings and the nitrogen positions can be picked out of the layout.

smiles = 'Cn1c(=O)c2c(ncn2C)n(C)c1=O'
mol = Chem.MolFromSmiles(smiles)

G_mol = nx.Graph()
for atom in mol.GetAtoms():
    G_mol.add_node(atom.GetIdx(), element=atom.GetSymbol())
for bond in mol.GetBonds():
    G_mol.add_edge(bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())

print(f'Caffeine contains {G_mol.number_of_nodes()} nodes and '
      f'{G_mol.number_of_edges()} edges.')

element_to_number = {'C': 0, 'N': 1, 'O': 2}
node_values = [element_to_number.get(G_mol.nodes[n]['element'], 3) for n in G_mol.nodes]

plt.figure(figsize=(9, 7))
pos = nx.spring_layout(G_mol, seed=42)
nx.draw(G_mol, pos, with_labels=True, node_size=1000, node_color=node_values,
        width=2, font_size=10, font_color='blue',)
plt.title('Caffeine as a molecular graph')
plt.axis('off')
plt.show()


The graph is more complicated, the vocabulary is not:

- nodes are atoms
- edges are bonds
- node features are atom properties
- edge features are bond properties
- the graph-level property belongs to the whole molecule


### 6.10 Why graphs, and what a model does with them

Why not simply put the atoms in a table, the way Part 1 put `FEATURE_COLS` in a table? Two
problems.

**Molecules have different numbers of atoms.** One complex has 30 atoms, the next has 70. A
fixed-width row cannot hold a variable number of atoms without padding or discarding. The
tabular baselines in Part 1 got around this by summarising each complex into counts and
averages, which works but throws away everything about arrangement.

**Connectivity matters.** These two graphs use the same three atoms:

```text
A ----- B ----- C          A ----- C ----- B
```

They are different molecules. For chemistry, connectivity is not decoration, it is identity.

At the same time a graph is defined only by its connectivity, so `A - B - C` and `C - B - A`
are the same graph. Redrawing a molecule, or renumbering its atoms, must not change the
prediction, and a graph-based model is built so that it does not.

**What a GNN does with it.** Each atom starts with its own feature vector, then repeatedly
updates that vector using the vectors of its neighbors. After one round an atom knows about
the atoms one bond away, after two rounds about the atoms two bonds away, and so on:

```text
Round 0:   [C]
Round 1:   C - [C] - O
Round 2:   C - C - [C] - O
```

The atom representations are finally pooled into one vector for the whole graph, and that
vector is what predicts the redox potential. Day 4 builds this.



**Mentor checkpoint 8**: after the graph section

- What do the nodes and edges of a molecular graph represent, and what does the adjacency
  matrix add to the picture?
- Why can a fixed-width table of atom properties not represent molecules of different sizes?
- Which of the node features listed in 6.6 would you expect to matter for redox potential?
- How does the degree of the iron node relate to the coordination numbers counted in 5.4?

Proceed only after confirmation.

---


## Exercises



### Exercise 1 - Hyperparameter tuning with GridSearchCV (Required)

1. Define a parameter grid with at least 3 hyperparameters for Random Forest.
2. Use 5-fold cross-validation.
3. Report the best parameters and improvement over the default model.

A starter grid is provided in the notebook as commented code; students should uncomment and run it.

### Exercise 2 - Learning curve analysis

1. Train the Random Forest with increasing fractions of the training data (10%, 20%, ..., 100%).
2. Plot train and test RMSE vs. training set size.
3. Discuss whether more data would likely improve performance.

### Exercise 3 - Build the graph of a real complex

1. Take the `mol` object from any row of `df`.
2. Use `rdkit` to draw the complex
3. Build a NetworkX graph from its atoms and bonds, as in 6.5.
4. Report the number of nodes and edges, and the degree of the iron atom.
5. Check that degree against the coordination numbers counted in 5.4.

### Exercise 4 - Count and plot ligand denticity

**Denticity** is the number of donor atoms of a *single* ligand that are bonded to the metal
centre. A ligand binding through one atom is monodentate (ammonia, chloride), through two is
bidentate (ethylenediamine, oxalate), through three tridentate, and so on; a ligand that
grips the metal at several points at once is called a chelate. Part 5 sorted ligands by what
they are made of, so this exercise sorts them by how they attach instead.

Denticity and the coordination number of 5.4 are two views of the same thing: the
coordination number of an iron centre is the sum of the denticities of its ligands. Six
monodentate ligands and three bidentate ones both give CN 6.

**Counting it.** Section 5.1 already deletes the iron and splits the rest into fragments. One
piece is missing: after `RemoveAtom` the atom indices are renumbered, so there is no way to
tell which fragment atoms used to touch the iron. Fix that by labelling the atoms first.

1. Find the iron and record the indices of its neighbors *before* removing it:

   ```python
   fe_idx = next((a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == 'Fe'), None)
   neighbor_idxs = {n.GetIdx() for n in mol.GetAtomWithIdx(fe_idx).GetNeighbors()}
   ```

2. Tag every atom with its original index, so the label survives fragmentation:

   ```python
   for atom in mol.GetAtoms():
       atom.SetIntProp('origIdx', atom.GetIdx())
   ```

3. Remove the iron with `Chem.RWMol` and split with
   `Chem.GetMolFrags(mol_wo_fe, asMols=True, sanitizeFrags=False)`, exactly as in 5.1.

4. For each fragment, the denticity is the number of its atoms whose `origIdx` is in
   `neighbor_idxs`:

   ```python
   denticity = sum(1 for a in frag.GetAtoms()
                   if a.HasProp('origIdx') and a.GetIntProp('origIdx') in neighbor_idxs)
   ```

   A fragment with denticity 0 was never bonded to the metal: a counter-ion or a trapped
   solvent molecule, not a ligand. Skip it.

5. Turn the fragment into a canonical SMILES so the same ligand from different complexes
   collapses to one key. `Chem.RemoveHs`, then `Chem.SanitizeMol` inside a `try` (breaking the
   metal bonds can leave aromaticity in a state RDKit will not accept), then `MolToSmiles`.

6. Accumulate over the whole dataset in a `defaultdict(list)` keyed by SMILES, appending each
   observed denticity. The same ligand does not always bind the same way, so give each unique
   ligand one representative value: `int(round(np.mean(counts)))`.

**Plotting it.** A histogram of one integer per unique ligand, built the same way as the
coordination number figure in 5.4:

```python
sns.histplot(unique_ligand_denticities, discrete=True, color='darkcyan', shrink=0.7)
```

Annotate each bar with its percentage of the total, force integer x-ticks with
`plt.xticks(sorted(counts_series.index))`, and label the y-axis "Count of unique ligands", not
"Count of ligands": one row per distinct ligand, not per occurrence. Compare with the pie
chart in 5.2, which counts occurrences, and be clear about which of the two you are showing.

**Checks and questions.**

1. Nothing should exceed denticity 6. Print any ligand that does, along with its list of
   observed denticities, and work out what went wrong with that structure.
2. For a few complexes, verify that the denticities of the ligands sum to the iron
   coordination number from 5.4.
3. What fraction of unique ligands are monodentate? Does that match the impression the pie
   chart in 5.2 gives?
4. Cross the denticity against the classes from 5.1. Which classes chelate?
5. Split the dataset by whether a complex contains a chelating ligand and compare the redox
   potential distributions. Is there a difference, and is it larger than the spread within
   each group?

---



## Summary

| Model | Strengths | Weaknesses | Best For |
|---|---:|---|---|
| Random Forest | Non-linear, feature importance, robust | No uncertainty, can't extrapolate | Medium datasets, feature selection |
| Gaussian Process Regression | Uncertainty estimates, principled | O(n^3) scaling, kernel choice | Small datasets, uncertainty needed |
| GNN (Day 4) | Learns from structure, transferable | Needs more data, complex | Molecular property prediction |

| Graph concept | Molecular meaning | Where it appeared |
|---|---|---|
| Node | Atom | 6.4, 6.6 |
| Edge | Bond | 6.4 |
| Node features | Atom properties: $Z$, charge, degree, aromaticity | 6.6 |
| Edge features | Bond properties: order, aromaticity, ring membership | 6.3 |
| Adjacency matrix | Connectivity as a numerical array | 6.7 |
| Degree | Number of bonded neighbors | 6.8, and Fe coordination in 5.4 |
| Neighborhood | The atoms one bond away, the unit a GNN passes messages over | 6.8 |
| Graph-level target | Redox potential of the whole complex | 6.3 |

---



## Preparation for Day 4

Day 4 will cover GNNs trained on molecular graphs using PyTorch Geometric on GPU nodes. Before Day 4 ensure:

- `baseline_results.csv` is present in the repository to compare GNN performance against classical baselines.
- Review the feature shortlist determined in Mentor checkpoint 6; these features will be used as baselines and for ablation studies.



